# Stage 1 v3 — DANN + 5-Fold Subject CV  (Colab Pro / Pro+ variant)

Same code, same training contract as the Kaggle notebook. Differences:
- Paths default to `/content/` (Colab convention).
- Optional Google Drive mount so the cache, manifest, and incremental `stage1v3_results.json` survive across sessions.
- GPU type printout so you know whether you got T4 / L4 / A100.

**Training contract (locked from Stage 1 v2)**: optimiser, scheduler, augmentation, seed, Conformer config — UNCHANGED across all 15 runs. The only variable is the signer head (alpha ∈ {0, 1.0, 0.3}).

**Expected wall-clock**:
| GPU | Sweep time |
|---|---|
| T4 (Colab free / Pro) | ~10 h |
| L4 (Pro typical)      | ~4 h  |
| A100 (Pro best case)  | ~2.4 h |

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy --quiet

import sys, os
# Colab gives a writable root at /content. /tmp would also work; /content
# is the convention so File-Browser sidebar shows everything.
BASE = '/content'
WITA_ROOT = os.path.join(BASE, 'wita_v2')

# Fresh clone every session (Colab containers are ephemeral by default).
!rm -rf {WITA_ROOT}
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" {WITA_ROOT}
sys.path.insert(0, BASE)

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU            : {name}  ({mem:.1f} GB)')
    if 'A100' in name:
        print('Estimated sweep time: ~2.4 h')
    elif 'L4' in name:
        print('Estimated sweep time: ~4 h')
    elif 'T4' in name:
        print('Estimated sweep time: ~10 h \u2014 consider switching to L4/A100 in Runtime > Change runtime type.')
    else:
        print('Estimated sweep time: depends on this GPU.')

## Cell 2 — Optional Drive mount, then config

Mounting Drive persists the skeleton cache, CV manifest, and incremental `stage1v3_results.json` across sessions. Skip the mount and you'll regenerate the cache (~30 min) on every fresh runtime.

Set `USE_DRIVE = False` if you don't want to authorise Drive access.

In [ ]:
import os, logging, random, json
import numpy as np
import torch

USE_DRIVE = True       # set False to keep everything in /content/

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PERSIST_ROOT = '/content/drive/MyDrive/wita_v2_stage1v3'
else:
    PERSIST_ROOT = '/content/wita_v2_stage1v3'

# Persistent (survives runtime resets if Drive is mounted)
os.makedirs(PERSIST_ROOT, exist_ok=True)
SKELETON_CACHE = os.path.join(PERSIST_ROOT, 'skeleton_features_t32.pt')
CV_MANIFEST    = os.path.join(PERSIST_ROOT, 'subject_cv5.json')
RESULTS_PATH   = os.path.join(PERSIST_ROOT, 'stage1v3_results.json')

# Ephemeral per-session (still writeable; we don't strictly need persistence here)
EPHEM = '/content'
os.makedirs(os.path.join(EPHEM, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(EPHEM, 'logs'),        exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s \u2014 %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(os.path.join(EPHEM, 'logs', 'stage1v3.log')),
    ],
)

from wita_v2.configs.default import (
    Config, DataConfig, AugConfig, EncoderConfig,
    RecurrentConfig, AttnDecoderConfig, TrainConfig,
)

# Stage 1 v2 hyperparams \u2014 LOCKED. Do not edit per-variant.
T_NATIVE       = 32
UPSAMPLE       = 2
D_MODEL        = 256
N_LAYERS       = 4
N_HEADS        = 4
CONV_KERNEL    = 15
DROPOUT        = 0.2
BATCH_SIZE     = 32        # locked: changing this violates plan \u00a712
LR_PEAK        = 5e-4
WEIGHT_DECAY   = 5e-2
GRAD_CLIP      = 1.0
NUM_EPOCHS     = 80
WARMUP_PCT     = 0.05
SEED           = 42
GAMMA_GRL      = 10.0

VARIANTS = [
    {'name': 'no_dann',  'alpha': 0.0},
    {'name': 'dann_a1',  'alpha': 1.0},
    {'name': 'dann_a03', 'alpha': 0.3},
]
FOLDS = list(range(5))

cfg = Config(
    data=DataConfig(
        hf_repo_id='yewon816/WiTA',
        lang='english', max_zips=None, max_frames=64,
        train_split=0.90, seed=SEED,
        download_dir=os.path.join(EPHEM, 'hf_downloads'),
    ),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr=LR_PEAK,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
        num_workers=2, warmup_pct=WARMUP_PCT, seed=SEED,
        checkpoint_dir=os.path.join(EPHEM, 'checkpoints'),
    ),
).build()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

print(f'Device         : {cfg.device}')
print(f'Persist root   : {PERSIST_ROOT}')
print(f'Cache target   : {SKELETON_CACHE}')
print(f'Results target : {RESULTS_PATH}')
print(f'Variants       : {[v["name"] for v in VARIANTS]}')
print(f'Folds          : {FOLDS}')
print(f'Locked from v2 : dropout={DROPOUT}  wd={WEIGHT_DECAY}  lr={LR_PEAK}  batch={BATCH_SIZE}')

## Cell 3 — Stream + index data (skip if Stage 1 cache already exists)

In [ ]:
from wita_v2.datasets.subject_splits import stream_and_index_with_subjects
if os.path.exists(SKELETON_CACHE):
    print(f'Cache exists at {SKELETON_CACHE} \u2014 skipping stream/extraction.')
    samples = None      # we don't need it once the cache is built
else:
    samples = stream_and_index_with_subjects(cfg)
    print(f'Total: {len(samples)} clips across {len({s for _,_,s in samples})} subjects')

## Cell 4 — Build (or load) skeleton cache

In [ ]:
from wita_v2.datasets.skeleton_cache import extract_skeleton_features
if not os.path.exists(SKELETON_CACHE):
    extract_skeleton_features(
        samples=samples, out_path=SKELETON_CACHE,
        T_native=T_NATIVE, dtype=torch.float16,
    )
cache = torch.load(SKELETON_CACHE, map_location='cpu', weights_only=False)
print(f'Loaded cache: {len(cache["feats"])} clips, '
      f'detect_rate={cache.get("frame_detect_rate", 0)*100:.1f}%')

## Cell 5 — Build 5-fold subject CV manifest

In [ ]:
from wita_v2.datasets.cv_splits import build_cv5_manifest, save_cv5_manifest, load_cv5_manifest
from wita_v2.datasets.subject_splits import stream_and_index_with_subjects

if os.path.exists(CV_MANIFEST):
    manifest = load_cv5_manifest(CV_MANIFEST)
    print(f'Loaded CV manifest from {CV_MANIFEST}')
else:
    # Need the subject list from the SAMPLES (or from the cache).
    # Use cache subjects which were stored at extraction time.
    fake_samples = [(b'', cache['labels'][i], cache['subjects'][i])
                    for i in range(len(cache['feats']))]
    manifest = build_cv5_manifest(fake_samples, n_folds=5, seed=SEED)
    save_cv5_manifest(manifest, CV_MANIFEST)

print(f'\nCV manifest: {manifest["n_subjects_total"]} signers, {manifest["n_folds"]} folds')
for f in manifest['folds']:
    print(f'  fold {f["fold"]}: train={f["n_train_subjects"]} signers ({f["n_train_clips"]} clips)  '
          f'val={f["n_val_subjects"]} signers ({f["n_val_clips"]} clips)')
    print(f'    val subjects: {f["val_subjects"]}')

## Cell 6 — Run the 15-run sweep

Each fold-variant call uses the same `train_one_fold` function from `training/dann_train.py`. Results are written to disk incrementally (and to Drive if mounted) so the notebook is robust to Colab session disconnects.

Loop order is **fold-major, variant-minor**: after every 3 completed runs you have one fully comparable fold across all 3 variants. If T4 only finishes ~10 of 15 runs in 8 h, the partial result is still publishable as 3 complete folds, and the resume logic picks up the remaining folds next session.

In [ ]:
from wita_v2.training.dann_train import train_one_fold
from wita_v2.datasets.cv_splits  import fold_indices, build_fold_signer_map
from wita_v2.datasets.skeleton_augment import LandmarkAugment

# Resume support: load existing results if the run was interrupted.
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    completed = {(r['fold'], r['variant']) for r in all_results}
    print(f'Resuming \u2014 {len(completed)} runs already complete.')
else:
    all_results = []
    completed = set()

train_aug = LandmarkAugment()          # same defaults as Stage 1 v2

subject_per_idx = cache['subjects']

for fold in FOLDS:
    train_idx, val_idx = fold_indices(manifest, fold, subject_per_idx)
    signer_map = build_fold_signer_map(manifest, fold)

    for v in VARIANTS:
        if (fold, v['name']) in completed:
            continue
        result = train_one_fold(
            cache=cache,
            train_idx=train_idx, val_idx=val_idx,
            signer_to_local=signer_map,
            cfg=cfg,
            fold=fold, variant=v['name'], alpha=v['alpha'],
            gamma_grl=GAMMA_GRL,
            num_epochs=NUM_EPOCHS,
            batch_size=BATCH_SIZE,
            lr_peak=LR_PEAK,
            weight_decay=WEIGHT_DECAY,
            grad_clip=GRAD_CLIP,
            dropout=DROPOUT,
            d_model=D_MODEL,
            n_layers=N_LAYERS,
            n_heads=N_HEADS,
            conv_kernel=CONV_KERNEL,
            upsample=UPSAMPLE,
            warmup_pct=WARMUP_PCT,
            transform=train_aug,
        )
        # strip history for the top-level results file
        summary = {k: v for k, v in result.items() if k != 'history'}
        all_results.append(summary)
        with open(RESULTS_PATH, 'w') as f:
            json.dump(all_results, f, indent=2)
        print(f'  saved \u2192 {RESULTS_PATH}  (total runs done: {len(all_results)})\n')

print(f'\nAll {len(all_results)}/{len(VARIANTS)*len(FOLDS)} runs done.')

## Cell 7 — Aggregate: mean ± std per variant, paired Wilcoxon test

In [ ]:
import numpy as np
from scipy.stats import wilcoxon

with open(RESULTS_PATH) as f:
    all_results = json.load(f)

by_variant = {v['name']: [None]*len(FOLDS) for v in VARIANTS}
for r in all_results:
    by_variant[r['variant']][r['fold']] = r['best_val_cer']

print('Per-fold best val CER\n')
header = ' fold  ' + '  '.join(f'{v["name"]:>10s}' for v in VARIANTS)
print(header); print('-'*len(header))
for fold in FOLDS:
    row = f' {fold:>4d}  '
    for v in VARIANTS:
        c = by_variant[v['name']][fold]
        row += f'  {c:>10.4f}' if c is not None else f'  {"-":>10s}'
    print(row)

print('\nMean \u00b1 std across folds:')
stats = {}
for v in VARIANTS:
    arr = np.array([c for c in by_variant[v['name']] if c is not None])
    stats[v['name']] = (arr.mean(), arr.std(ddof=0))
    print(f'  {v["name"]:>10s}: {arr.mean():.4f} \u00b1 {arr.std(ddof=0):.4f}  (n={len(arr)})')

# Paired Wilcoxon: DANN variants vs no_dann
no = by_variant['no_dann']
for vname in ['dann_a1', 'dann_a03']:
    pairs = [(a, b) for a, b in zip(no, by_variant[vname]) if a is not None and b is not None]
    if len(pairs) < 2:
        print(f'\nSkipping {vname}: not enough paired observations.')
        continue
    a = np.array([p[0] for p in pairs]); b = np.array([p[1] for p in pairs])
    try:
        stat, p = wilcoxon(a, b, zero_method='zsplit', alternative='two-sided')
        print(f'\nPaired Wilcoxon  no_dann vs {vname}:')
        print(f'  diff (no_dann - {vname}): mean={float((a-b).mean()):+.4f}  '
              f'sign(+ = DANN helps): {int((a > b).sum())}/{len(a)} folds')
        print(f'  W={stat:.3f}  two-sided p={p:.4f}')
    except ValueError as e:
        print(f'\nWilcoxon failed for {vname}: {e}')

## Cell 8 — Verdict per plan §5

In [ ]:
no_mean, no_std = stats['no_dann']
b_mean,  b_std  = stats['dann_a1']
delta = b_mean - no_mean

print('\n=== Stage 1 v3 verdict ===')
print(f'  variant A (no DANN) : {no_mean:.4f} \u00b1 {no_std:.4f}')
print(f'  variant B (DANN \u03b1=1): {b_mean:.4f} \u00b1 {b_std:.4f}')
print(f'  \u0394 (B - A)         : {delta:+.4f}')
print()
if delta <= -0.05:
    print('  \u2705 H1 CONFIRMED \u2014 bake DANN into Stages 3\u201310.')
elif -0.05 < delta <= -0.01:
    print('  \u26a1 H1 weakly confirmed \u2014 keep DANN as an ablation row.')
elif abs(delta) <= 0.01:
    print('  \u274c H1 FALSIFIED \u2014 signer shift is not the bottleneck. Audit MediaPipe, '
          'try bigger Conformer, try CTC-MixUp.')
else:
    print('  \u26a0  DANN destabilised training \u2014 lower \u03b1 to 0.1, retry one fold.')
print()
if no_std >= 0.04:
    print(f'  \u2705 H2 CONFIRMED (std {no_std:.4f} \u2265 0.04) \u2014 single-split numbers are fragile. '
          'Report all future stages with 5-fold mean \u00b1 std.')
elif no_std <= 0.015:
    print(f'  \u2139 H2 not confirmed (std {no_std:.4f} \u2264 0.015) \u2014 single-split may be OK for '
          'routine ablations; keep CV for the final matrix.')
else:
    print(f'  \u2139 borderline std {no_std:.4f} \u2014 report mean\u00b1std on headline rows.')

## Cell 9 — Per-signer val CER scatter (H3 outlier detector)

If DANN works, it should help more on signer-outlier folds. The scatter below shows per-signer best-epoch CER across all 39 signers, colored by variant.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

per_signer: dict[str, dict[str, float]] = {}
for r in all_results:
    for sid, cer in r.get('best_per_signer_val_cer', {}).items():
        per_signer.setdefault(r['variant'], {})[sid] = cer

all_signers = sorted(set().union(*[d.keys() for d in per_signer.values()]))
colors = {'no_dann': '#1f77b4', 'dann_a1': '#d62728', 'dann_a03': '#2ca02c'}

fig, ax = plt.subplots(figsize=(11, 4.5))
for vname, d in per_signer.items():
    xs = list(range(len(all_signers)))
    ys = [d.get(s, float('nan')) for s in all_signers]
    ax.scatter(xs, ys, label=vname, s=30, alpha=0.75, color=colors.get(vname, None))
ax.set_xticks(range(len(all_signers)))
ax.set_xticklabels(all_signers, rotation=90, fontsize=7)
ax.set_ylabel('Val CER at best-epoch')
ax.set_title('Per-signer val CER across variants (1 signer = 1 CV fold)')
ax.grid(True, linestyle=':', alpha=0.3)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(EPHEM, 'logs', 'stage1v3_per_signer_scatter.png'), dpi=120)
plt.show()